# Interaction Model Examples

<a href="http://35.236.121.59/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fproject-chip%2Fconnectedhomeip&urlpath=lab%2Ftree%2Fconnectedhomeip%2Fdocs%2Fguides%2Frepl%2FMatter%2520-%2520Basic%2520Interactions.ipynb&branch=master">
<img src="https://i.ibb.co/hR3yWsC/launch-playground.png" alt="drawing" width="130"/>
</a>
<br></br>

This walks through the various interactions that can be initiated from the REPL towards a target using the Matter Interaction Model (IM) and Data Model (DM).

## Clear Persisted Storage

Let's clear out our persisted storage (if one exists) to start from a clean slate.

In [ ]:
import os, subprocess

if os.path.isfile('/tmp/repl-storage.json'):
    os.remove('/tmp/repl-storage.json')

# So that the all-clusters-app won't boot with stale prior state.
os.system('rm -rf /tmp/chip_*')

## Initialization

Let's first begin by setting up by importing some key modules that are needed to make it easier for us to interact with the Matter stack.

`ChipReplStartup.py` is run within the global namespace. This results in all of its imports being made available here.

> **NOTE**: _This is not needed if you launch the REPL from the command-line._

In [32]:
%reset -f
import importlib.util
spec = importlib.util.find_spec('chip.ChipReplStartup')
%run {spec.origin}

 Replacing  store path ./credentials/development/paa-root-certs with 
/Users/badra/opt/test/connectedhomeip/credentials/development/paa-root-certs
Note that you are still running from /Users/badra/opt/test/connectedhomeip/docs/development_controllers/chip-repl 
so other relative paths may be off.

─────────────────────────────────────────────────── Matter REPL ───────────────────────────────────────────────────

            Welcome to the Matter Python REPL!

            For help, please type matterhelp()

            To get more information on a particular object/class, you can pass
            that into matterhelp() as well.

            

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

The following objects have been created:

certificateAuthorityManager:    Manages a list of CertificateAuthority instances.
        caList:                         The list of CertificateAuthority instances.
        caList[n].adminList[m]:         A specific FabricAdmin object at index m for the nth CertificateAuthority 
instance.

Default CHIP Device Controller (NodeId: 112233): has been initialized to manage caList[0].adminList[0] (FabricId = 
1), and is available as devCtrl

#### Commission the Water Heater Device change the descrimator amd pincode as per the device on-boarding codes

In [44]:
# _ = await devCtrl.CommissionWiFi(872,72012440, 1, 'PhotonSmart', 'Photon@2024')
_ = await devCtrl.CommissionWiFi(1506,39708778, 1, 'PhotonSmart', 'Photon@2024')
#_ = await devCtrl.CommissionWiFi(3422,56706809, 2, 'PhotonSmart', 'Photon@2024')

In [49]:
await devCtrl.SendCommand(1,1,Clusters.WaterHeaterMode.Commands.ChangeToMode(1))


ChangeToModeResponse(
│   status=0,
│   statusText=None
)

In [24]:
import time
for i in range(30):
    await devCtrl.SendCommand(2,1,Clusters.WaterHeaterMode.Commands.ChangeToMode(1))
    time.sleep(0.001)
    await devCtrl.SendCommand(2,1,Clusters.WaterHeaterMode.Commands.ChangeToMode(2))
    time.sleep(0.001)
    await devCtrl.SendCommand(2,1,Clusters.WaterHeaterMode.Commands.ChangeToMode(3))
    time.sleep(0.001)


In [171]:
from datetime import datetime
import time


AtomicResponse(
│   statusCode=0,
│   attributeStatus=[
│   │   AtomicAttributeStatusStruct(
│   │   │   attributeID=81,
│   │   │   statusCode=0
│   │   )
│   ],
│   timeout=60000
)

In [186]:
now = datetime.now()
minutes_now = now.hour * 60 + now.minute

await devCtrl.SendCommand(1,1,Clusters.Thermostat.Commands.AtomicRequest(0,[81],60000))
temperature = 5300
trans = []
trans.append(Clusters.Thermostat.Structs.ScheduleTransitionStruct(dayOfWeek=0x7F,transitionTime=(minutes_now+1),heatingSetpoint=temperature,systemMode=4))
trans.append(Clusters.Thermostat.Structs.ScheduleTransitionStruct(dayOfWeek=0x7F,transitionTime=(minutes_now+2),heatingSetpoint=temperature+100,systemMode=4))
trans.append(Clusters.Thermostat.Structs.ScheduleTransitionStruct(dayOfWeek=0x7F,transitionTime=(minutes_now+3),heatingSetpoint=temperature+200,systemMode=4))

schedule = Clusters.Thermostat.Structs.ScheduleStruct(systemMode=4,transitions=trans)
await devCtrl.WriteAttribute(1, [ (1, Clusters.Thermostat.Attributes.Schedules([schedule]))])
await devCtrl.SendCommand(1,1,Clusters.Thermostat.Commands.AtomicRequest(1,[81]))


AtomicResponse(
│   statusCode=0,
│   attributeStatus=[
│   │   AtomicAttributeStatusStruct(
│   │   │   attributeID=81,
│   │   │   statusCode=0
│   │   )
│   ],
│   timeout=None
)

In [11]:
# await devCtrl.ReadAttribute(1, [
#                                  (1, Clusters.Thermostat.Attributes.ActiveScheduleHandle),
#                                  (1, Clusters.Thermostat.Attributes.NumberOfScheduleTransitionPerDay),
#                                  (1, Clusters.Thermostat.Attributes.NumberOfScheduleTransitions), 
#                                  (1, Clusters.Thermostat.Attributes.OccupiedHeatingSetpoint), 
#                                  (0, Clusters.BasicInformation.Attributes.SoftwareVersionString),
#                                  (0, Clusters.BasicInformation.Attributes.SoftwareVersion)
#                                ])
await devCtrl.ReadAttribute(1, [(1,Clusters.FreshWaterHeaterController)])


{
│   1: {
│   │   <class 'chip.clusters.Objects.FreshWaterHeaterController'>: {
│   │   │   <class 'chip.clusters.Attribute.DataVersion'>: 938158187,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.ClusterRevision'>: 1,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.MaximumBoostTime'>: 7200,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.AntiLegionellaState'>: <AntiLegionellaStateEnum.kInactive: 0>,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.ColdWaterTemperature'>: 2000,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.ShowerState'>: <ShowerStateEnum.kActive: 1>,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.EcoModeSetpoint'>: 4500,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.DisplayActiveTimeout'>: 15000,
│   │   │   <class 'chip.clusters.Objects.Fres

In [236]:
await devCtrl.SendCommand(1,1,Clusters.Thermostat.Commands.SetActiveScheduleRequest(b''))
# await devCtrl.SendCommand(1,1,Clusters.Thermostat.Commands.SetActiveScheduleRequest(b'\x01'))

In [181]:
def parse_day_of_week(day_of_week_bitmap):
    """
    Parse dayOfWeek bitmap to day names.
    Bit 0 = Sunday, Bit 1 = Monday, ..., Bit 6 = Saturday
    """
    days = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
    active_days = []
    
    for i, day in enumerate(days):
        if day_of_week_bitmap & (1 << i):
            active_days.append(day)
    
    return active_days
def parse_transition_time(transition_time, use_24hr=False):
    """
    Parse transitionTime to hours and minutes.
    transitionTime is in minutes since midnight.
    
    Args:
        transition_time: Minutes since midnight
        use_24hr: If True, use 24-hour format. If False, use 12-hour AM/PM format.
    """
    hours = transition_time // 60
    minutes = transition_time % 60
    
    if use_24hr:
        return f"{hours:02d}:{minutes:02d}"
    else:
        # Convert to 12-hour format with AM/PM
        period = "AM" if hours < 12 else "PM"
        hours_12 = hours % 12
        if hours_12 == 0:
            hours_12 = 12
        return f"{hours_12:02d}:{minutes:02d} {period}"


In [237]:
schedulesData = await devCtrl.ReadAttribute(1, [(1, Clusters.Thermostat.Attributes.Schedules)])
transitionsParsed = schedulesData[1][chip.clusters.Objects.Thermostat][chip.clusters.Objects.Thermostat.Attributes.Schedules][0].transitions
for transition in transitionsParsed:
    print("--------------------------------------")
    print(f"Days of Week: {parse_day_of_week(transition.dayOfWeek)}")
    print(f"Transition Time: {parse_transition_time(transition.transitionTime)}")
    print(f"Heating Setpoint: {transition.heatingSetpoint}")

--------------------------------------
Days of Week: ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
Transition Time: 12:13 AM
Heating Setpoint: 5300
--------------------------------------
Days of Week: ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
Transition Time: 12:14 AM
Heating Setpoint: 5400
--------------------------------------
Days of Week: ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
Transition Time: 12:15 AM
Heating Setpoint: 5500


In [11]:
import time
help(time.sleep)


Help on built-in function sleep in module time:

sleep(...)
    sleep(seconds)
    
    Delay execution for a given number of seconds.  The argument may be
    a floating point number for subsecond precision.



In [19]:
await devCtrl.SendCommand(1,1,Clusters.WaterHeaterMode.Commands.ChangeToMode(2))
time.sleep(1)
boostInfo = Clusters.WaterHeaterManagement.Structs.WaterHeaterBoostInfoStruct(duration=30, oneShot=True,temporarySetpoint=7000)
await devCtrl.SendCommand(1,1,Clusters.WaterHeaterManagement.Commands.Boost(boostInfo))

In [13]:
await devCtrl.SendCommand(1,1,Clusters.WaterHeaterManagement.Commands.CancelBoost())

In [59]:
await devCtrl.SendCommand(1,1,Clusters.FreshWaterHeaterController.Commands.AnodeChangeRequest())

In [58]:
await devCtrl.SendCommand(1,1,Clusters.FreshWaterHeaterController.Commands.AnodeChangeConfirmed())

###### await devCtrl.WriteAttribute(1, [(1,Clusters.FreshWaterHeaterController.Attributes.DiagnosticsConfirmTimeList([180000, 240000, 0, 180000, 120000, 180000, 0]))])
await devCtrl.ReadAttribute(1, [
    (1, Clusters.FreshWaterHeaterController.Attributes.DiagnosticsConfirmTimeList),
])

In [ ]:
await devCtrl.WriteAttribute(1, [(1,Clusters.FreshWaterHeaterController.Attributes.

In [54]:
# DISPLAY‑RELATED ATTRIBUTES TEST
#  - DisplayActiveTimeout
#  - DisplayErrorTimeout
#  - DisplayTargetTimeout
#  - DisplayTemperatureStep
import time
# 1) READ current values
await devCtrl.ReadAttribute(1, [
    (1, Clusters.FreshWaterHeaterController.Attributes.DisplayActiveTimeout),
    (1, Clusters.FreshWaterHeaterController.Attributes.DisplayErrorTimeout),
    (1, Clusters.FreshWaterHeaterController.Attributes.DisplayTargetTimeout),
    (1, Clusters.FreshWaterHeaterController.Attributes.DisplayTemperatureStep),
])

# 2) WRITE test values (adjust if your product has different limits)
await devCtrl.WriteAttribute(1, [
    (1, Clusters.FreshWaterHeaterController.Attributes.DisplayActiveTimeout(25000)),   # 5 s
    (1, Clusters.FreshWaterHeaterController.Attributes.DisplayErrorTimeout(20000)),    # 3 s
    (1, Clusters.FreshWaterHeaterController.Attributes.DisplayTargetTimeout(5000)),   # 4 s
    (1, Clusters.FreshWaterHeaterController.Attributes.DisplayTemperatureStep(400)),  # 5.00 °C
])

time.sleep(1)

In [55]:
# 3) READ back to verify they stuck
await devCtrl.ReadAttribute(1, [
    (1, Clusters.FreshWaterHeaterController.Attributes.DisplayActiveTimeout),
    (1, Clusters.FreshWaterHeaterController.Attributes.DisplayErrorTimeout),
    (1, Clusters.FreshWaterHeaterController.Attributes.DisplayTargetTimeout),
    (1, Clusters.FreshWaterHeaterController.Attributes.DisplayTemperatureStep),
])


{
│   1: {
│   │   <class 'chip.clusters.Objects.FreshWaterHeaterController'>: {
│   │   │   <class 'chip.clusters.Attribute.DataVersion'>: 933701622,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.DisplayActiveTimeout'>: 25000,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.DisplayErrorTimeout'>: 20000,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.DisplayTemperatureStep'>: 400,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.DisplayTargetTimeout'>: 5000
│   │   }
│   }
}

In [65]:
import time

# 1) READ current values
await devCtrl.ReadAttribute(1, [
    (1, Clusters.FreshWaterHeaterController.Attributes.CoolDownTimeout),
    (1, Clusters.FreshWaterHeaterController.Attributes.ResetTimeout),
    (1, Clusters.FreshWaterHeaterController.Attributes.ResetCounterTimeout),
    (1, Clusters.FreshWaterHeaterController.Attributes.MaximumBoostTime),
])


{
│   1: {
│   │   <class 'chip.clusters.Objects.FreshWaterHeaterController'>: {
│   │   │   <class 'chip.clusters.Attribute.DataVersion'>: 1712673369,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.MaximumBoostTime'>: 30,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.ResetTimeout'>: 10,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.CoolDownTimeout'>: 6000,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.ResetCounterTimeout'>: 7
│   │   }
│   }
}

In [70]:
# 3) READ back to confirm
# 2) WRITE test values (all in seconds)
await devCtrl.WriteAttribute(1, [
    (1, Clusters.FreshWaterHeaterController.Attributes.CoolDownTimeout(6000)),      # 30 s
    (1, Clusters.FreshWaterHeaterController.Attributes.ResetTimeout(9)),          # 7 s
    (1, Clusters.FreshWaterHeaterController.Attributes.ResetCounterTimeout(5)),   # 7 s
    (1, Clusters.FreshWaterHeaterController.Attributes.MaximumBoostTime(30)),   # 2 h (match AL boost)
])

# time.sleep(1)

# await devCtrl.ReadAttribute(1, [
#     (1, Clusters.FreshWaterHeaterController.Attributes.CoolDownTimeout),
#     (1, Clusters.FreshWaterHeaterController.Attributes.ResetTimeout),
#     (1, Clusters.FreshWaterHeaterController.Attributes.ResetCounterTimeout),
#     (1, Clusters.FreshWaterHeaterController.Attributes.MaximumBoostTime),
# ])


[
│   AttributeStatus(
│   │   Path=AttributePath(
│   │   │   EndpointId=1,
│   │   │   ClusterId=367524869,
│   │   │   AttributeId=10
│   │   ),
│   │   Status=<Status.Success: 0>
│   ),
│   AttributeStatus(
│   │   Path=AttributePath(
│   │   │   EndpointId=1,
│   │   │   ClusterId=367524869,
│   │   │   AttributeId=9
│   │   ),
│   │   Status=<Status.Success: 0>
│   ),
│   AttributeStatus(
│   │   Path=AttributePath(
│   │   │   EndpointId=1,
│   │   │   ClusterId=367524869,
│   │   │   AttributeId=11
│   │   ),
│   │   Status=<Status.Success: 0>
│   ),
│   AttributeStatus(
│   │   Path=AttributePath(
│   │   │   EndpointId=1,
│   │   │   ClusterId=367524869,
│   │   │   AttributeId=25
│   │   ),
│   │   Status=<Status.Success: 0>
│   )
]

In [34]:
boostInfo = Clusters.WaterHeaterManagement.Structs.WaterHeaterBoostInfoStruct(duration=3000, oneShot=True,temporarySetpoint=7000)
await devCtrl.SendCommand(1,1,Clusters.WaterHeaterManagement.Commands.Boost(boostInfo))

InteractionModelError: InteractionModelError: InvalidInState (0xcb)

In [62]:
boostInfo = Clusters.WaterHeaterManagement.Structs.WaterHeaterBoostInfoStruct(duration=10, oneShot=True,temporarySetpoint=7000)
await devCtrl.SendCommand(1,1,Clusters.WaterHeaterManagement.Commands.Boost(boostInfo))

In [61]:
boostInfo = Clusters.WaterHeaterManagement.Structs.WaterHeaterBoostInfoStruct(duration=300, oneShot=True,temporarySetpoint=7000)
await devCtrl.SendCommand(1,1,Clusters.WaterHeaterManagement.Commands.Boost(boostInfo))

InteractionModelError: InteractionModelError: InvalidInState (0xcb)

In [38]:
boostInfo = Clusters.WaterHeaterManagement.Structs.WaterHeaterBoostInfoStruct(duration=20, oneShot=True,temporarySetpoint=7000)
await devCtrl.SendCommand(1,1,Clusters.WaterHeaterManagement.Commands.Boost(boostInfo))

2025-11-30 20:24:41 Mohameds-MacBook-Pro.local chip.native.EM[19856] ERROR <<5 [E:8380i S:36857 M:222668795] (S) Msg Retransmission to 1:0000000000000001 failure (max retries:4)


ChipStackError: src/app/CommandSender.cpp:354: CHIP Error 0x00000032: Timeout

In [ ]:
import time

# 1) READ current values
await devCtrl.ReadAttribute(1, [
    (1, Clusters.FreshWaterHeaterController.Attributes.ColdWaterTemperature),
    (1, Clusters.WaterHeaterManagement.Attributes.TankPercentage),
    (1, Clusters.FreshWaterHeaterController.Attributes.StandardModeSetpoint),
    (1, Clusters.FreshWaterHeaterController.Attributes.EcoModeSetpoint),
    (1, Clusters.FreshWaterHeaterController.Attributes.OverheatThresholdTemperature),
    (1, Clusters.FreshWaterHeaterController.Attributes.HeaterMaximumPower),
])

In [57]:
# 2) WRITE test values  (adjust if needed to stay within your limits)
await devCtrl.WriteAttribute(1, [
    (1, Clusters.FreshWaterHeaterController.Attributes.ColdWaterTemperature(2000)),   # 15.00 °C
    (1, Clusters.FreshWaterHeaterController.Attributes.StandardModeSetpoint(6500)),  # 60.00 °C
    (1, Clusters.FreshWaterHeaterController.Attributes.EcoModeSetpoint(5000)),       # 45.00 °C
    (1, Clusters.FreshWaterHeaterController.Attributes.OverheatThresholdTemperature(9500)),  # 80.00 °C
    (1, Clusters.2HeaterMaximumPower(2000)),    # 2 kW
])

time.sleep(2)

# 3) READ back to confirm the write
await devCtrl.ReadAttribute(1, [
    (1, Clusters.FreshWaterHeaterController.Attributes.ColdWaterTemperature),
    (1, Clusters.WaterHeaterManagement.Attributes.TankPercentage),
    (1, Clusters.FreshWaterHeaterController.Attributes.StandardModeSetpoint),
    (1, Clusters.FreshWaterHeaterController.Attributes.EcoModeSetpoint),
    (1, Clusters.FreshWaterHeaterController.Attributes.OverheatThresholdTemperature),
    (1, Clusters.FreshWaterHeaterController.Attributes.HeaterMaximumPower),
])


{
│   1: {
│   │   <class 'chip.clusters.Objects.FreshWaterHeaterController'>: {
│   │   │   <class 'chip.clusters.Attribute.DataVersion'>: 4135381250,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.EcoModeSetpoint'>: 5000,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.HeaterMaximumPower'>: 2000,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.StandardModeSetpoint'>: 6500,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.OverheatThresholdTemperature'>: 9500,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.ColdWaterTemperature'>: 2000
│   │   },
│   │   <class 'chip.clusters.Objects.WaterHeaterManagement'>: {
│   │   │   <class 'chip.clusters.Attribute.DataVersion'>: 780783143,
│   │   │   <class 'chip.clusters.Objects.WaterHeaterManagement.Attributes.TankPercentage'>: 100
│   │   }
│   }
}

In [62]:
await devCtrl.WriteAttribute(1, [
    (1, Clusters.FreshWaterHeaterController.Attributes.HeaterMaximumPower(1500))])


[
│   AttributeStatus(
│   │   Path=AttributePath(
│   │   │   EndpointId=1,
│   │   │   ClusterId=367524869,
│   │   │   AttributeId=21
│   │   ),
│   │   Status=<Status.Success: 0>
│   )
]

In [71]:
import time

# 1) READ current shower-related attributes
await devCtrl.ReadAttribute(1, [
    (1, Clusters.FreshWaterHeaterController.Attributes.ShowerTemperature),
    (1, Clusters.FreshWaterHeaterController.Attributes.ShowerHysteresis),
    (1, Clusters.FreshWaterHeaterController.Attributes.ShowerState),
])


{
│   1: {
│   │   <class 'chip.clusters.Objects.FreshWaterHeaterController'>: {
│   │   │   <class 'chip.clusters.Attribute.DataVersion'>: 2232479431,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.ShowerTemperature'>: 60,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.ShowerHysteresis'>: 7,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.ShowerState'>: <ShowerStateEnum.kInactive: 0>
│   │   }
│   }
}

In [86]:
# 2) WRITE test values (adjust if needed)
await devCtrl.WriteAttribute(1, [
    (1, Clusters.FreshWaterHeaterController.Attributes.ShowerTemperature(6000)),   # 42.00 °C
    (1, Clusters.FreshWaterHeaterController.Attributes.ShowerHysteresis(700)),     # 2.00 °C 
])

time.sleep(2)

# 3) READ back to confirm the write
await devCtrl.ReadAttribute(1, [
    (1, Clusters.FreshWaterHeaterController.Attributes.ShowerTemperature),
    (1, Clusters.FreshWaterHeaterController.Attributes.ShowerHysteresis),
    (1, Clusters.FreshWaterHeaterController.Attributes.ShowerState),
])


{
│   1: {
│   │   <class 'chip.clusters.Objects.FreshWaterHeaterController'>: {
│   │   │   <class 'chip.clusters.Attribute.DataVersion'>: 1999988569,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.ShowerTemperature'>: 6000,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.ShowerHysteresis'>: 700,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.ShowerState'>: <ShowerStateEnum.kInactive: 0>
│   │   }
│   }
}

In [95]:
await devCtrl.WriteAttribute(1, [
    (1, Clusters.FreshWaterHeaterController.Attributes.PreviousTargetHeaterTemperature(60)),   # 42.00 °C
])


[
│   AttributeStatus(
│   │   Path=AttributePath(
│   │   │   EndpointId=1,
│   │   │   ClusterId=367524869,
│   │   │   AttributeId=20
│   │   ),
│   │   Status=<Status.UnsupportedWrite: 136>
│   )
]

In [102]:
await devCtrl.ReadAttribute(1, [
    (1, Clusters.FreshWaterHeaterController.Attributes.ErrorCode),
    (1, Clusters.FreshWaterHeaterController.Attributes.RapidRiseDelta),
    (1, Clusters.FreshWaterHeaterController.Attributes.RapidRiseWindow),
])


{
│   1: {
│   │   <class 'chip.clusters.Objects.FreshWaterHeaterController'>: {
│   │   │   <class 'chip.clusters.Attribute.DataVersion'>: 1712583337,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.RapidRiseDelta'>: 500,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.RapidRiseWindow'>: 20,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.ErrorCode'>: 32
│   │   }
│   }
}

In [103]:
import time

# 1) READ current values
await devCtrl.ReadAttribute(1, [
    (1, Clusters.FreshWaterHeaterController.Attributes.RapidRiseDelta),
    (1, Clusters.FreshWaterHeaterController.Attributes.RapidRiseWindow),
    (1, Clusters.FreshWaterHeaterController.Attributes.TemperatureSensorMinValid),
    (1, Clusters.FreshWaterHeaterController.Attributes.TemperatureSensorMaxValid),
])


{
│   1: {
│   │   <class 'chip.clusters.Objects.FreshWaterHeaterController'>: {
│   │   │   <class 'chip.clusters.Attribute.DataVersion'>: 1712583337,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.RapidRiseDelta'>: 500,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.RapidRiseWindow'>: 20,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.TemperatureSensorMinValid'>: 0,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.TemperatureSensorMaxValid'>: 10000
│   │   }
│   }
}

In [122]:
# 2) WRITE test values
# RapidRiseDelta/Window are in centi‑deg C and seconds
# Min/Max valid are in centi‑deg C
await devCtrl.WriteAttribute(1, [
    (1, Clusters.FreshWaterHeaterController.Attributes.RapidRiseDelta(1000)),     # 5.00 °C delta
    (1, Clusters.FreshWaterHeaterController.Attributes.RapidRiseWindow(30)),     # 20 s window
    (1, Clusters.FreshWaterHeaterController.Attributes.TemperatureSensorMinValid(0)),     # 0.00 °C
    (1, Clusters.FreshWaterHeaterController.Attributes.TemperatureSensorMaxValid(10000)), # 100.00 °C
])

time.sleep(2)

# 3) READ back to confirm
await devCtrl.ReadAttribute(1, [
    (1, Clusters.FreshWaterHeaterController.Attributes.RapidRiseDelta),
    (1, Clusters.FreshWaterHeaterController.Attributes.RapidRiseWindow),
    (1, Clusters.FreshWaterHeaterController.Attributes.TemperatureSensorMinValid),
    (1, Clusters.FreshWaterHeaterController.Attributes.TemperatureSensorMaxValid),
])


{
│   1: {
│   │   <class 'chip.clusters.Objects.FreshWaterHeaterController'>: {
│   │   │   <class 'chip.clusters.Attribute.DataVersion'>: 3481579549,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.RapidRiseDelta'>: 1000,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.RapidRiseWindow'>: 30,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.TemperatureSensorMinValid'>: 0,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.TemperatureSensorMaxValid'>: 10000
│   │   }
│   }
}

In [121]:
await devCtrl.WriteAttribute(1, [(1,Clusters.FreshWaterHeaterController.Attributes.DiagnosticsConfirmTimeList([10000, 5000, 0, 180000, 120000, 180000, 0]))])
await devCtrl.ReadAttribute(1, [
    (1, Clusters.FreshWaterHeaterController.Attributes.DiagnosticsConfirmTimeList),
])


{
│   1: {
│   │   <class 'chip.clusters.Objects.FreshWaterHeaterController'>: {
│   │   │   <class 'chip.clusters.Attribute.DataVersion'>: 3481579540,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.DiagnosticsConfirmTimeList'>: [
│   │   │   │   10000,
│   │   │   │   5000,
│   │   │   │   0,
│   │   │   │   180000,
│   │   │   │   120000,
│   │   │   │   180000,
│   │   │   │   0
│   │   │   ]
│   │   }
│   }
}

In [112]:
await devCtrl.ReadAttribute(1, [
    (1, Clusters.FreshWaterHeaterController.Attributes.DiagnosticsConfirmTimeList),
])


{
│   1: {
│   │   <class 'chip.clusters.Objects.FreshWaterHeaterController'>: {
│   │   │   <class 'chip.clusters.Attribute.DataVersion'>: 1886807893,
│   │   │   <class 'chip.clusters.Objects.FreshWaterHeaterController.Attributes.DiagnosticsConfirmTimeList'>: [
│   │   │   │   1000,
│   │   │   │   240000,
│   │   │   │   0,
│   │   │   │   180000,
│   │   │   │   120000,
│   │   │   │   180000,
│   │   │   │   0
│   │   │   ]
│   │   }
│   }
}

In [12]:
await devCtrl.ReadAttribute(1, [
    (2, Clusters.ElectricalPowerMeasurement),
])


{
│   2: {
│   │   <class 'chip.clusters.Objects.ElectricalPowerMeasurement'>: {
│   │   │   <class 'chip.clusters.Attribute.DataVersion'>: 858085740,
│   │   │   <class 'chip.clusters.Objects.ElectricalPowerMeasurement.Attributes.ClusterRevision'>: 1,
│   │   │   <class 'chip.clusters.Objects.ElectricalPowerMeasurement.Attributes.RMSCurrent'>: Null,
│   │   │   <class 'chip.clusters.Objects.ElectricalPowerMeasurement.Attributes.Accuracy'>: [
│   │   │   │   MeasurementAccuracyStruct(
│   │   │   │   │   measurementType=<MeasurementTypeEnum.kActivePower: 5>,
│   │   │   │   │   measured=True,
│   │   │   │   │   minMeasuredValue=0,
│   │   │   │   │   maxMeasuredValue=2000000,
│   │   │   │   │   accuracyRanges=[
│   │   │   │   │   │   MeasurementAccuracyRangeStruct(
│   │   │   │   │   │   │   rangeMin=0,
│   │   │   │   │   │   │   rangeMax=99999,
│   │   │   │   │   │   │   percentMax=None,
│   │   │   │   │   │   │   percentMin=None,
│   │   │   │   │   │   │   percentTypical=Non

In [35]:
devCtrl.SendCommand(2, 1, FreshWaterHeaterController.Commands.AnodeChangeRequest())

NameError: name 'FreshWaterHeaterController' is not defined